In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt


In [6]:
files = {
    "RackA": "../data/rack_a_telemetry.csv",
    "RackB": "../data/rack_b_telemetry.csv",
    "RackC": "../data/rack_c_telemetry.csv",
    "RackD": "../data/rack_d_telemetry.csv"
}




In [8]:
dfs = []

for rack, path in files.items():
    df = pd.read_csv(path)
    
    # Force correct datetime parsing (handles +00:00)
    df["time_utc"] = pd.to_datetime(df["time_utc"], utc=True, errors="coerce")
    
    df["rack_id"] = rack
    dfs.append(df)


In [9]:
data = (
    pd.concat(dfs, ignore_index=True)
      .dropna(subset=["time_utc"])
      .sort_values(by="time_utc")
      .reset_index(drop=True)
)

data.head()


,time_utc,ambient_temp_c,cpu_util,exhaust_temp_c,fan_speed_rpm,humidity,inlet_temp_c,power_kw,rack_id
0,2025-12-14 07:17:20.685000+00:00,25.29,79.38,34.92,4382.0,46.52,29.26,1.29,RackA
1,2025-12-14 07:18:28.804000+00:00,25.90,38.56,33.55,3157.0,45.84,27.83,0.89,RackA
2,2025-12-14 07:18:38.785000+00:00,25.49,76.35,34.78,4290.0,48.12,29.30,1.26,RackA
3,2025-12-14 07:18:48.804000+00:00,25.47,45.92,33.31,3378.0,49.08,27.76,0.96,RackA
4,2025-12-14 07:18:59.007000+00:00,25.62,40.66,33.43,3220.0,49.78,27.66,0.91,RackA


In [10]:
## Sanity Check
print(data.isna().sum())
data.describe()


time_utc          0
ambient_temp_c    0
cpu_util          0
exhaust_temp_c    0
fan_speed_rpm     0
humidity          0
inlet_temp_c      0
power_kw          0
rack_id           0
dtype: int64


,ambient_temp_c,cpu_util,exhaust_temp_c,fan_speed_rpm,humidity,inlet_temp_c,power_kw
count,7811.000000,7811.000000,7811.000000,7811.000000,7811.000000,7811.000000,7811.000000
mean,25.308693,60.491822,37.105829,4465.455511,46.289547,29.141384,1.858395
std,0.582979,17.668126,4.048762,1243.319984,1.943674,2.028719,0.951279
min,24.000000,25.060000,31.580000,2900.000000,42.000000,25.440000,0.800000
25%,25.000000,45.695000,34.250000,3541.500000,44.890000,27.680000,1.190000
50%,25.370000,60.460000,35.460000,3979.000000,46.260000,28.520000,1.430000
75%,25.750000,74.605000,38.520000,4949.500000,47.845000,30.150000,2.015000
max,26.500000,95.000000,47.000000,7000.000000,50.000000,34.030000,3.700000


In [11]:
## Features
FEATURES = [
    "cpu_util",
    "power_kw",
    "ambient_temp_c",
    "inlet_temp_c",
    "exhaust_temp_c",
    "fan_speed_rpm",
    "humidity"
]


In [14]:
## Scaling
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

data_scaled = data.copy()

for rack in data_scaled["rack_id"].unique():
    idx = data_scaled["rack_id"] == rack
    data_scaled.loc[idx, FEATURES] = scaler.fit_transform(
        data_scaled.loc[idx, FEATURES]
    )


In [15]:
data_scaled[FEATURES].describe()


,cpu_util,power_kw,ambient_temp_c,inlet_temp_c,exhaust_temp_c,fan_speed_rpm,humidity
count,7811.000000,7811.000000,7811.000000,7811.000000,7811.000000,7811.000000,7811.000000
mean,0.497341,0.497481,0.496921,0.495659,0.497608,0.497373,0.505742
std,0.288052,0.288098,0.289218,0.216531,0.188555,0.288051,0.286983
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.246716,0.240000,0.246667,0.338384,0.360958,0.246871,0.260000
50%,0.498799,0.500000,0.500000,0.494929,0.497696,0.498980,0.505000
75%,0.745279,0.740260,0.750000,0.656716,0.634895,0.745165,0.752000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [17]:
##Sequence Builder (Sliding Window 5 minutes)

def create_sequences(df, features, window=30):
    X, y = [], []
    for i in range(len(df) - window):
        X.append(df[features].iloc[i:i+window].values)
        y.append(df[features].iloc[i+window].values)
    return np.array(X), np.array(y)


In [18]:
X_all, y_all = [], []

WINDOW = 30

for rack in data_scaled["rack_id"].unique():
    rack_df = data_scaled[data_scaled["rack_id"] == rack]
    
    X, y = create_sequences(rack_df, FEATURES, WINDOW)
    
    X_all.append(X)
    y_all.append(y)

X = np.vstack(X_all)
y = np.vstack(y_all)

print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (7691, 30, 7)
y shape: (7691, 7)


In [19]:
## sanity check
# Check one sample
print(X[0].shape)
print(y[0])

# Ensure no NaNs
print(np.isnan(X).sum(), np.isnan(y).sum())


(30, 7)
[0.27221777 0.28       0.89       0.44674556 0.38018433 0.27218145
 0.238     ]
0 0


In [20]:
##“Feature normalization and sequence construction were performed independently for each rack to preserve localized operational dynamics and prevent cross-rack data leakage.”

In [21]:
np.save("X.npy", X)
np.save("y.npy", y)
